# Pipeline Validation System

This notebook validates your current backend pipeline tables using the real source-backed table flow.

It covers:
- required table existence
- required columns
- critical non-null checks
- uniqueness/grain checks
- hierarchical structural null checks for C2G
- coherence failure checks
- `next_3m_forecast` cross-table parity
- C2G additivity checks

It also saves results to:
- `retail.qa_validation_results`
- `retail.qa_validation_summary`

In [19]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = "postgres"
DB_PASSWORD = "Nestle123"
DB_HOST = "127.0.0.1"
DB_PORT = "5432"
DB_NAME = "nestle_forecasting"
SCHEMA = "retail"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    future=True,
    pool_pre_ping=True,
)


def read_df(query: str, parse_dates=None) -> pd.DataFrame:
    return pd.read_sql_query(text(query), engine, parse_dates=parse_dates)


def scalar(query: str):
    with engine.begin() as conn:
        return conn.execute(text(query)).scalar()


def table_exists(table_name: str, schema: str = SCHEMA) -> bool:
    q = """
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = :schema
          AND table_name = :table_name
    )
    """
    with engine.begin() as conn:
        return bool(
            conn.execute(
                text(q),
                {"schema": schema, "table_name": table_name},
            ).scalar()
        )


def get_columns(table_name: str, schema: str = SCHEMA) -> list[str]:
    q = """
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = :schema
      AND table_name = :table_name
    ORDER BY ordinal_position
    """
    with engine.begin() as conn:
        rows = conn.execute(
            text(q),
            {"schema": schema, "table_name": table_name},
        ).fetchall()
    return [r[0] for r in rows]


def quote_table(table_name: str, schema: str = SCHEMA) -> str:
    return f'"{schema}"."{table_name}"'


def count_rows(table_name: str, schema: str = SCHEMA) -> int:
    return int(scalar(f"SELECT COUNT(*) FROM {quote_table(table_name, schema)}") or 0)


def append_result(
    results,
    check_type: str,
    table_name: str,
    check_name: str,
    status: str,
    details: str = "",
    severity: str = "error",
) -> None:
    results.append(
        {
            "check_type": check_type,
            "table_name": table_name,
            "check_name": check_name,
            "status": status,
            "severity": severity,
            "details": details,
        }
    )


def run_required_table_checks(results, required_tables):
    for t in required_tables:
        if not table_exists(t):
            append_result(
                results,
                "table_exists",
                t,
                "required table exists",
                "FAIL",
                "table missing",
            )
        else:
            n = count_rows(t)
            append_result(
                results,
                "table_exists",
                t,
                "required table exists",
                "PASS",
                f"rows={n:,}",
            )

            if t == "api_coherence_failed_checks":
                row_status = "PASS" if n == 0 else "WARN"
            else:
                row_status = "PASS" if n > 0 else "WARN"

            append_result(
                results,
                "row_count",
                t,
                "table has rows",
                row_status,
                f"rows={n:,}",
                severity="warning",
            )


def run_required_column_checks(results, spec):
    for table_name, required_cols in spec.items():
        if not table_exists(table_name):
            append_result(
                results,
                "required_columns",
                table_name,
                "required columns present",
                "SKIP",
                "table missing",
                severity="warning",
            )
            continue

        cols = set(get_columns(table_name))
        missing = []

        for req in required_cols:
            if isinstance(req, (list, tuple, set)):
                if not any(alias in cols for alias in req):
                    missing.append(list(req))
            else:
                if req not in cols:
                    missing.append(req)

        if missing:
            append_result(
                results,
                "required_columns",
                table_name,
                "required columns present",
                "FAIL",
                f"missing={missing}",
            )
        else:
            append_result(
                results,
                "required_columns",
                table_name,
                "required columns present",
                "PASS",
                "all required/alias columns found",
            )


def run_non_null_checks(results, spec):
    for table_name, cols in spec.items():
        if not table_exists(table_name):
            append_result(
                results,
                "nonnull",
                table_name,
                "critical non-null fields",
                "SKIP",
                "table missing",
                severity="warning",
            )
            continue

        existing = set(get_columns(table_name))

        for col in cols:
            if isinstance(col, (list, tuple, set)):
                actual_col = next((alias for alias in col if alias in existing), None)
                label = " / ".join(col)
            else:
                actual_col = col if col in existing else None
                label = col

            if actual_col is None:
                append_result(
                    results,
                    "nonnull",
                    table_name,
                    f"{label} is non-null",
                    "SKIP",
                    "column missing",
                    severity="warning",
                )
                continue

            q = f'SELECT COUNT(*) FROM {quote_table(table_name)} WHERE "{actual_col}" IS NULL'
            n = int(scalar(q) or 0)

            append_result(
                results,
                "nonnull",
                table_name,
                f"{label} is non-null",
                "PASS" if n == 0 else "FAIL",
                f"null_count={n:,}",
            )


def run_unique_checks(results, spec):
    for table_name, cols in spec.items():
        if not table_exists(table_name):
            append_result(
                results,
                "uniqueness",
                table_name,
                "unique grain",
                "SKIP",
                "table missing",
                severity="warning",
            )
            continue

        existing = set(get_columns(table_name))
        resolved = []
        missing = []

        for col in cols:
            if isinstance(col, (list, tuple, set)):
                actual_col = next((alias for alias in col if alias in existing), None)
                if actual_col is None:
                    missing.append(list(col))
                else:
                    resolved.append(actual_col)
            else:
                if col not in existing:
                    missing.append(col)
                else:
                    resolved.append(col)

        if missing:
            append_result(
                results,
                "uniqueness",
                table_name,
                "unique grain",
                "SKIP",
                f"missing columns={missing}",
                severity="warning",
            )
            continue

        expr = ", ".join([f'"{c}"' for c in resolved])
        q = f"""
        SELECT COUNT(*) FROM (
            SELECT {expr}, COUNT(*) AS n
            FROM {quote_table(table_name)}
            GROUP BY {expr}
            HAVING COUNT(*) > 1
        ) d
        """
        dup_groups = int(scalar(q) or 0)

        append_result(
            results,
            "uniqueness",
            table_name,
            f"unique on {resolved}",
            "PASS" if dup_groups == 0 else "FAIL",
            f"duplicate_groups={dup_groups:,}",
        )


def run_c2g_structural_null_checks(results):
    table_name = "api_c2g_top_contributors"

    if not table_exists(table_name):
        append_result(
            results,
            "structural_nulls",
            table_name,
            "C2G structural null validation",
            "SKIP",
            "table missing",
            severity="warning",
        )
        return

    cols_needed = {
        "level",
        "nestle_region",
        "nestle_store_cluster",
        "store_code",
        "product_code",
    }
    existing = set(get_columns(table_name))

    if not cols_needed.issubset(existing):
        append_result(
            results,
            "structural_nulls",
            table_name,
            "C2G structural null validation",
            "SKIP",
            "missing required columns",
            severity="warning",
        )
        return

    rules = [
        ("Region", "nestle_region", True),
        ("Region", "nestle_store_cluster", False),
        ("Region", "store_code", False),
        ("Region", "product_code", False),
        ("Cluster", "nestle_region", True),
        ("Cluster", "nestle_store_cluster", True),
        ("Cluster", "store_code", False),
        ("Cluster", "product_code", False),
        ("Store", "nestle_region", True),
        ("Store", "nestle_store_cluster", True),
        ("Store", "store_code", True),
        ("Store", "product_code", False),
        ("SKU", "nestle_region", True),
        ("SKU", "nestle_store_cluster", True),
        ("SKU", "store_code", True),
        ("SKU", "product_code", True),
        ("National", "nestle_region", False),
        ("National", "nestle_store_cluster", False),
        ("National", "store_code", False),
        ("National", "product_code", False),
    ]

    for level, col, should_be_present in rules:
        comparator = "IS NULL" if should_be_present else "IS NOT NULL"
        q = f"""
        SELECT COUNT(*) FROM {quote_table(table_name)}
        WHERE "level" = :level AND "{col}" {comparator}
        """
        with engine.begin() as conn:
            n = int(conn.execute(text(q), {"level": level}).scalar() or 0)

        check_name = (
            f'{table_name}: {col} '
            f'{"present" if should_be_present else "structurally null"} '
            f"at level={level}"
        )

        append_result(
            results,
            "structural_nulls",
            table_name,
            check_name,
            "PASS" if n == 0 else "FAIL",
            f"violations={n:,}",
        )


def run_coherence_checks(results):
    table_name = "api_coherence_failed_checks"

    if not table_exists(table_name):
        append_result(
            results,
            "coherence",
            table_name,
            "coherence failures table exists",
            "SKIP",
            "table missing",
            severity="warning",
        )
        return

    n = count_rows(table_name)
    append_result(
        results,
        "coherence",
        table_name,
        "coherence failures count is zero",
        "PASS" if n == 0 else "FAIL",
        f"rows={n:,}",
    )


def run_next_3m_check(results):
    if not table_exists("api_overview_kpis") or not table_exists("api_forecast_horizon"):
        append_result(
            results,
            "cross_table",
            "api_overview_kpis",
            "next_3m_forecast backfill parity",
            "SKIP",
            "required tables missing",
            severity="warning",
        )
        return

    kpi_cols = set(get_columns("api_overview_kpis"))
    horizon_cols = set(get_columns("api_forecast_horizon"))

    if (
        "next_3m_forecast" not in kpi_cols
        or "reconciled_forecast" not in horizon_cols
        or "ds" not in horizon_cols
    ):
        append_result(
            results,
            "cross_table",
            "api_overview_kpis",
            "next_3m_forecast backfill parity",
            "SKIP",
            "required columns missing",
            severity="warning",
        )
        return

    parse_cols = ["as_of_month"] if "as_of_month" in kpi_cols else None
    kpis = read_df(
        f'SELECT * FROM {quote_table("api_overview_kpis")}',
        parse_dates=parse_cols,
    )
    horizon = read_df(
        f'SELECT ds, reconciled_forecast FROM {quote_table("api_forecast_horizon")} ORDER BY ds',
        parse_dates=["ds"],
    )

    expected = (
        pd.to_numeric(horizon["reconciled_forecast"], errors="coerce")
        .fillna(0)
        .head(3)
        .sum()
        if len(horizon)
        else np.nan
    )
    actual = (
        pd.to_numeric(kpis["next_3m_forecast"], errors="coerce").iloc[0]
        if len(kpis) and "next_3m_forecast" in kpis.columns
        else np.nan
    )

    if pd.isna(expected) and pd.isna(actual):
        status = "PASS"
    elif pd.isna(expected) != pd.isna(actual):
        status = "FAIL"
    else:
        status = "PASS" if abs(float(actual) - float(expected)) < 1e-6 else "FAIL"

    append_result(
        results,
        "cross_table",
        "api_overview_kpis",
        "next_3m_forecast backfill parity",
        status,
        f"expected={expected}, actual={actual}",
    )


def run_c2g_additivity_checks(results):
    table_name = "mart_c2g_all_levels"

    if not table_exists(table_name):
        append_result(
            results,
            "c2g_additivity",
            table_name,
            "C2G shares sum correctly",
            "SKIP",
            "table missing",
            severity="warning",
        )
        return

    needed = {"ds", "level", "c2g_vs_ly", "c2g_vs_last"}
    existing = set(get_columns(table_name))

    if not needed.issubset(existing):
        append_result(
            results,
            "c2g_additivity",
            table_name,
            "C2G shares sum correctly",
            "SKIP",
            "required columns missing",
            severity="warning",
        )
        return

    df = read_df(
        f"""
        SELECT ds, level,
               SUM(c2g_vs_ly) AS sum_c2g_vs_ly,
               SUM(c2g_vs_last) AS sum_c2g_vs_last
        FROM {quote_table(table_name)}
        GROUP BY ds, level
        ORDER BY ds, level
        """,
        parse_dates=["ds"],
    )

    tol = 1e-3
    for _, row in df.iterrows():
        ly = row["sum_c2g_vs_ly"]
        last = row["sum_c2g_vs_last"]

        ly_ok = pd.isna(ly) or abs(float(ly) - 1.0) <= tol
        last_ok = pd.isna(last) or abs(float(last) - 1.0) <= tol
        status = "PASS" if (ly_ok and last_ok) else "FAIL"

        append_result(
            results,
            "c2g_additivity",
            table_name,
            f'C2G shares sum to ~1 for ds={pd.Timestamp(row["ds"]).date()} level={row["level"]}',
            status,
            f"sum_c2g_vs_ly={ly}, sum_c2g_vs_last={last}, tol={tol}",
        )


def build_validation_results():
    results = []

    required_tables = [
        "stg_cleaned_sales_panel",
        "stg_feature_engineered_panel",
        "stg_mstl_product_month",
        "stg_mstl_decomposition_for_lightgbm",
        "stg_mstl_seasonal_profiles",
        "stg_lightgbm_base_forecasts",
        "stg_mint_reconciled_forecast",
        "mart_c2g_bottom_level",
        "mart_c2g_all_levels",
        "api_c2g_top_contributors",
        "api_overview_kpis",
        "api_forecast_horizon",
        "api_prescription_actions",
        "api_prescription_implementation_guide",
        "api_coherence_failed_checks",
    ]

    run_required_table_checks(results, required_tables)

    required_columns = {
        "stg_cleaned_sales_panel": [
            "ds",
            "Product Code",
            "Store Code",
            "NESTLE REGION",
            "NESTLE STORE CLUSTER",
            ["net_sales_ty", "Net Sales TY", "Net Sales TY (Ex-VAT)"],
        ],
        "stg_feature_engineered_panel": [
            "ds",
            ["product_code", "Product Code"],
            ["store_code", "Store Code"],
            ["nestle_region", "NESTLE REGION"],
            ["nestle_store_cluster", "NESTLE STORE CLUSTER"],
            "net_sales_ty",
            "split",
            "eligible_for_model",
        ],
        "stg_mstl_product_month": ["ds", "Product Code", "y"],
        "stg_mstl_decomposition_for_lightgbm": [
            "ds",
            ["product_code", "Product Code"],
            "trend",
            "seasonal_strength",
        ],
        "stg_mstl_seasonal_profiles": [
            ["product_code", "Product Code"],
            "month",
            "seasonal_multiplier",
        ],
        "stg_lightgbm_base_forecasts": [
            "ds",
            "product_code",
            "store_code",
            "base_forecast",
        ],
        "stg_mint_reconciled_forecast": [
            "ds",
            "product_code",
            "store_code",
            "nestle_region",
            "nestle_store_cluster",
            "reconciled_forecast",
        ],
        "mart_c2g_bottom_level": [
            "ds",
            "product_code",
            "store_code",
            "nestle_region",
            "nestle_store_cluster",
            "reconciled_forecast",
            "growth_vs_ly_abs",
            "c2g_vs_ly",
        ],
        "mart_c2g_all_levels": [
            "ds",
            "level",
            "reconciled_forecast",
            "growth_vs_ly_abs",
            "c2g_vs_ly",
        ],
        "api_c2g_top_contributors": [
            "ds",
            "level",
            "reconciled_forecast",
            "growth_vs_ly_abs",
            "c2g_vs_ly",
        ],
        "api_prescription_actions": [
            "priority",
            "condition_id",
            "action",
            "prescription_label",
            "action_guidance",
        ],
        "api_prescription_implementation_guide": [
            "sort_order",
            "action",
            "recommended_steps",
            "timeline",
        ],
    }

    run_required_column_checks(results, required_columns)

    critical_non_null = {
        "stg_cleaned_sales_panel": [
            "ds",
            "Product Code",
            "Store Code",
            "NESTLE REGION",
            "NESTLE STORE CLUSTER",
            ["net_sales_ty", "Net Sales TY", "Net Sales TY (Ex-VAT)"],
        ],
        "stg_feature_engineered_panel": [
            "ds",
            ["product_code", "Product Code"],
            ["store_code", "Store Code"],
            ["nestle_region", "NESTLE REGION"],
            ["nestle_store_cluster", "NESTLE STORE CLUSTER"],
            "net_sales_ty",
        ],
        "stg_mstl_product_month": ["ds", "Product Code", "y"],
        "stg_mstl_decomposition_for_lightgbm": [
            "ds",
            ["product_code", "Product Code"],
            "trend",
        ],
        "stg_mstl_seasonal_profiles": [
            ["product_code", "Product Code"],
            "month",
            "seasonal_multiplier",
        ],
        "stg_lightgbm_base_forecasts": [
            "ds",
            "product_code",
            "store_code",
            "base_forecast",
        ],
        "stg_mint_reconciled_forecast": [
            "ds",
            "product_code",
            "store_code",
            "nestle_region",
            "nestle_store_cluster",
            "reconciled_forecast",
        ],
        "api_overview_kpis": ["next_3m_forecast"],
        "api_forecast_horizon": ["ds", "reconciled_forecast"],
        "api_forecast_chart": ["ds"],
        "api_prescription_actions": [
            "priority",
            "condition_id",
            "action",
            "prescription_label",
        ],
    }

    run_non_null_checks(results, critical_non_null)

    unique_grains = {
        "stg_mstl_product_month": ["ds", "Product Code"],
        "stg_mstl_seasonal_profiles": [["product_code", "Product Code"], "month"],
        "stg_lightgbm_base_forecasts": ["ds", "product_code", "store_code"],
        "stg_mint_reconciled_forecast": ["ds", "product_code", "store_code"],
        "api_forecast_horizon": ["ds"],
        "api_prescription_implementation_guide": ["sort_order"],
    }

    run_unique_checks(results, unique_grains)
    run_c2g_structural_null_checks(results)
    run_coherence_checks(results)
    run_next_3m_check(results)
    run_c2g_additivity_checks(results)

    out = pd.DataFrame(results)

    if len(out):
        severity_order = {"error": 0, "warning": 1}
        status_order = {"FAIL": 0, "WARN": 1, "SKIP": 2, "PASS": 3}

        out["_sev"] = out["severity"].map(severity_order).fillna(9)
        out["_sta"] = out["status"].map(status_order).fillna(9)

        out = (
            out.sort_values(
                ["_sev", "_sta", "table_name", "check_type", "check_name"]
            )
            .drop(columns=["_sev", "_sta"])
            .reset_index(drop=True)
        )

    return out


def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(
            [{"status": "NO_RESULTS", "severity": "info", "count": 0}]
        )

    summary = (
        df.groupby(["status", "severity"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values(["severity", "status"])
        .reset_index(drop=True)
    )
    return summary


def save_results(
    df: pd.DataFrame,
    summary: pd.DataFrame,
    table_name: str = "qa_validation_results",
    summary_table_name: str = "qa_validation_summary",
    schema: str = SCHEMA,
) -> None:
    df.to_sql(
        table_name,
        engine,
        schema=schema,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=10000,
    )
    summary.to_sql(
        summary_table_name,
        engine,
        schema=schema,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=10000,
    )
    print(f"[ok] wrote {schema}.{table_name} and {schema}.{summary_table_name}")


def main(save_to_db: bool = True):
    results = build_validation_results()
    summary = summarize_results(results)

    print("\n=== VALIDATION SUMMARY ===")
    print(summary.to_string(index=False))

    if len(results):
        print("\n=== TOP FAILURES / WARNINGS ===")
        preview = results[results["status"].isin(["FAIL", "WARN"])].head(50)
        if len(preview):
            print(preview.to_string(index=False))
        else:
            print("No FAIL/WARN checks.")

    if save_to_db:
        save_results(results, summary)

    return results, summary


if __name__ == "__main__":
    main()


=== VALIDATION SUMMARY ===
status severity  count
  PASS    error    109
  PASS  warning     15

=== TOP FAILURES / WARNINGS ===
No FAIL/WARN checks.
[ok] wrote retail.qa_validation_results and retail.qa_validation_summary


In [20]:
results, summary = main(save_to_db=True)
results.head(20)


=== VALIDATION SUMMARY ===
status severity  count
  PASS    error    109
  PASS  warning     15

=== TOP FAILURES / WARNINGS ===
No FAIL/WARN checks.
[ok] wrote retail.qa_validation_results and retail.qa_validation_summary


,check_type,table_name,check_name,status,severity,details
0,required_columns,api_c2g_top_contributors,required columns present,PASS,error,all required/alias columns found
1,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_region presen...,PASS,error,violations=0
2,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_region presen...,PASS,error,violations=0
3,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_region presen...,PASS,error,violations=0
4,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_region presen...,PASS,error,violations=0
5,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_region struct...,PASS,error,violations=0
6,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_store_cluster...,PASS,error,violations=0
7,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_store_cluster...,PASS,error,violations=0
8,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_store_cluster...,PASS,error,violations=0
9,structural_nulls,api_c2g_top_contributors,api_c2g_top_contributors: nestle_store_cluster...,PASS,error,violations=0


In [7]:
results[results['status'].isin(['FAIL','WARN'])].head(100)

,check_type,table_name,check_name,status,severity,details
0,required_columns,mart_c2g_all_levels,required columns present,FAIL,error,missing=['driver_direction']
1,required_columns,stg_cleaned_sales_panel,required columns present,FAIL,error,missing=['net_sales_ty']
2,required_columns,stg_feature_engineered_panel,required columns present,FAIL,error,"missing=['product_code', 'store_code', 'nestle..."
3,required_columns,stg_mstl_decomposition_for_lightgbm,required columns present,FAIL,error,missing=['product_code']
4,required_columns,stg_mstl_seasonal_profiles,required columns present,FAIL,error,missing=['product_code']
113,row_count,api_coherence_failed_checks,table has rows,WARN,warning,rows=0


In [8]:
results.to_csv("qa_validation_results.csv", index=False)
summary.to_csv("qa_validation_summary.csv", index=False)
print("Saved qa_validation_results.csv and qa_validation_summary.csv")

Saved qa_validation_results.csv and qa_validation_summary.csv
